In [ ]:
!mkdir -p /data/sets/nuscenes  # Make the directory to store the nuScenes dataset in.

In [ ]:
!wget https://www.nuscenes.org/data/v1.0-mini.tgz  # Download the nuScenes mini split.

In [ ]:
!tar -xf v1.0-mini.tgz -C /data/sets/nuscenes  # Uncompress the nuScenes mini split.

In [ ]:
!pip uninstall -y numpy scipy scikit-learn
!pip install numpy==1.26.4 scipy==1.11.4 scikit-learn==1.3.2        # Install compatible numpy, scipy, and scikit-learn

In [ ]:
import os
os.kill(os.getpid(), 9)     # Restart runtime

In [ ]:
!pip install nuscenes-devkit        # Install nuscenes-devkit

In [ ]:
!pip install nuscenes-devkit ultralytics -q     # Install nuScenes-devkit ultralytics

In [ ]:
# nuScenes-devkit ultralytics check
from nuscenes.nuscenes import NuScenes
from ultralytics import YOLO
import numpy as np
print("✅ nuscenes-devkit OK")
print("✅ ultralytics OK")

In [ ]:
# Connect google drive 
from google.colab import drive
drive.mount('/content/drive')

NUSCENES_ROOT = '/data/sets/nuscenes'

import os
expected = ['samples', 'sweeps', 'maps', 'v1.0-mini']
for folder in expected:
  path = os.path.join(NUSCENES_ROOT, folder)
  status = "✅" if os.path.exists(path) else "missing"
  print(f"{status} {folder}/")

In [ ]:
nusc = NuScenes(version='v1.0-mini', dataroot=NUSCENES_ROOT, verbose=True)
my_sammple = nusc.sample[0]
nusc.render_sample(my_sammple['token'])         # NuScenes render_sample

In [ ]:
from PIL import Image
cam_token = my_sample['data']['CAM_FRONT']
cam_data = nusc.get('sample_data', cam_token)
img_path = os.path.join(NUSCENES_ROOT, cam_data['filename'])
print(f"Image path: {img_path}")
img = Image.open(img_path)
model = YOLO("yolo26n.pt")
results = model(img_path)
results[0].show()
print(f"✅ YOLO26 detected {len(results[0].boxes)} objects")        # yolo26n.pt model object detection on NuScenes sample 'CAM_FRONT' image test

In [ ]:
OUTPUT_DIR = '/content/nuscenes_yolo'

In [ ]:
# Create yaml file
yaml_content = f"""
path: {OUTPUT_DIR}
train: images/train
val: images/val

names:
  0: car
  1: pedestrian
  2: bicycle
  3: motorcycle
  4: bus
  5: truck
  6: traffic_cone
  7: barrier
"""

yaml_path = f'{OUTPUT_DIR}/nuscenes.yaml'
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print(open(yaml_path).read())
print(f"✅ Saved to {yaml_path}")

In [ ]:
# Nuscenes to YOLO Detection convertion
import os
import numpy as np
from nuscenes.nuscenes import NuScenes
from nuscenes.utils.geometry_utils import view_points, BoxVisibility
from nuscenes.utils.data_classes import Box
from pyquaternion import Quaternion
from pathlib import Path
import shutil
import random

NUSCENES_ROOT = '/data/sets/nuscenes'
OUTPUT_DIR = '/content/nuscenes_yolo'

CLASS_MAP = {
    'car': 0, 'pedestrian': 1, 'bicycle': 2, 'motorcycle': 3,
    'bus': 4, 'truck': 5, 'traffic_cone': 6, 'barrier': 7,
}

nusc = NuScenes(version='v1.0-mini', dataroot=NUSCENES_ROOT, verbose=False)

# Fresh output folders
if Path(OUTPUT_DIR).exists():
    shutil.rmtree(OUTPUT_DIR)
for split in ['train', 'val']:
    Path(f'{OUTPUT_DIR}/images/{split}').mkdir(parents=True, exist_ok=True)
    Path(f'{OUTPUT_DIR}/labels/{split}').mkdir(parents=True, exist_ok=True)

# Collect all samples
all_samples = []
for scene in nusc.scene:
    token = scene['first_sample_token']
    while token:
        all_samples.append(nusc.get('sample', token))
        token = nusc.get('sample', token)['next']

random.seed(42)
random.shuffle(all_samples)
split_idx = int(len(all_samples) * 0.8)
splits = {'train': all_samples[:split_idx], 'val': all_samples[split_idx:]}
print(f"Total: {len(all_samples)} → train: {split_idx}, val: {len(all_samples)-split_idx}")

converted, total_boxes = 0, 0

for split, samples in splits.items():
    for sample in samples:
        cam_token = sample['data']['CAM_FRONT']
        cam_data  = nusc.get('sample_data', cam_token)
        img_path  = os.path.join(NUSCENES_ROOT, cam_data['filename'])
        img_w, img_h = cam_data['width'], cam_data['height']

        # Camera calibration
        cs  = nusc.get('calibrated_sensor', cam_data['calibrated_sensor_token'])
        ep  = nusc.get('ego_pose', cam_data['ego_pose_token'])
        intrinsic = np.array(cs['camera_intrinsic'])

        # Get boxes in global frame, transform to camera frame
        boxes = nusc.get_boxes(cam_token)
        labels = []

        for box in boxes:
            # Map class
            cat_simple = next((k for k in CLASS_MAP if k in box.name), None)
            if cat_simple is None:
                continue

            # 1. Global → ego vehicle frame
            box.translate(-np.array(ep['translation']))
            box.rotate(Quaternion(ep['rotation']).inverse)

            # 2. Ego → camera frame
            box.translate(-np.array(cs['translation']))
            box.rotate(Quaternion(cs['rotation']).inverse)

            # 3. Skip if behind camera (z < 0.1 means too close or behind)
            if box.center[2] < 0.1:
                continue

            # 4. Project 3D corners to 2D image plane
            corners = view_points(box.corners(), intrinsic, normalize=True)
            xs, ys = corners[0], corners[1]

            x1, x2 = xs.min(), xs.max()
            y1, y2 = ys.min(), ys.max()

            # 5. Skip if completely outside image
            if x2 < 0 or x1 > img_w or y2 < 0 or y1 > img_h:
                continue

            # 6. Clip to image bounds
            x1 = np.clip(x1, 0, img_w)
            x2 = np.clip(x2, 0, img_w)
            y1 = np.clip(y1, 0, img_h)
            y2 = np.clip(y2, 0, img_h)

            # 7. Skip tiny or degenerate boxes
            if x2 - x1 < 4 or y2 - y1 < 4:
                continue
            if (x2 - x1) / img_w > 0.95 or (y2 - y1) / img_h > 0.95:
                continue

            # 8. YOLO format (normalized center + size)
            cx = (x1 + x2) / 2 / img_w
            cy = (y1 + y2) / 2 / img_h
            w  = (x2 - x1) / img_w
            h  = (y2 - y1) / img_h

            labels.append(f"{CLASS_MAP[cat_simple]} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
            total_boxes += 1

        # Save symlink + label
        img_name = Path(img_path).name
        dst = f'{OUTPUT_DIR}/images/{split}/{img_name}'
        if not os.path.exists(dst):
            os.symlink(img_path, dst)

        with open(f'{OUTPUT_DIR}/labels/{split}/{Path(img_name).stem}.txt', 'w') as f:
            f.write('\n'.join(labels))

        converted += 1

print(f"✅ Done: {converted} images, {total_boxes} total boxes")

# Verify
for split in ['train', 'val']:
    files = list(Path(f'{OUTPUT_DIR}/labels/{split}').glob('*.txt'))
    non_empty = sum(1 for f in files if f.stat().st_size > 0)
    print(f"  {split}: {len(files)} files, {non_empty} non-empty")

# Spot check — should see reasonable values (not 1.0)
from pathlib import Path
sample = next(f for f in Path(f'{OUTPUT_DIR}/labels/train').glob('*.txt') if f.stat().st_size > 0)
print(f"\nSample label ({sample.name}):")
print(open(sample).read()[:300])

In [ ]:
OUTPUT_DIR = '/content/drive/MyDrive/nuscenes_yolo'
RUNS_DIR = '/content/drive/MyDrive/yolo_runs'       # Set MyDrive as trained model save path

In [ ]:
# Train YOLO26 Detection Model
from ultralytics import YOLO

model = YOLO("yolo26n.pt")

model.train(
    data=yaml_path,
    epochs=100,
    imgsz=640,
    batch=16,
    name="yolo26n_nuscenes",
    project=RUNS_DIR,
    device=0,
    workers=2,
    patience=20,
    save=True,
    plots=True,
)

In [ ]:
import os
import numpy as np
import shutil
from pathlib import Path
from scipy.spatial import ConvexHull
from pyquaternion import Quaternion
from nuscenes.nuscenes import NuScenes
from nuscenes.utils.geometry_utils import view_points
import random

NUSCENES_ROOT = '/data/sets/nuscenes'
SEG_DIR       = '/content/drive/MyDrive/yolo_runs/nuscenes_yolo_seg'   # Save data to MyDrive

CLASS_MAP = {
    'car': 0, 'pedestrian': 1, 'bicycle': 2, 'motorcycle': 3,
    'bus': 4, 'truck': 5, 'traffic_cone': 6, 'barrier': 7,
}

# Min polygon points and min area thresholds
MIN_POLYGON_PTS = 4
MIN_AREA_RATIO  = 0.0001  # at least 0.01% of image area
MAX_AREA_RATIO  = 0.90    # at most 90% of image area

nusc = NuScenes(version='v1.0-mini', dataroot=NUSCENES_ROOT, verbose=False)

# Fresh output directories
if Path(SEG_DIR).exists():
    shutil.rmtree(SEG_DIR)
for split in ['train', 'val']:
    Path(f'{SEG_DIR}/images/{split}').mkdir(parents=True, exist_ok=True)
    Path(f'{SEG_DIR}/labels/{split}').mkdir(parents=True, exist_ok=True)

# Collect all samples and split 80/20
all_samples = []
for scene in nusc.scene:
    token = scene['first_sample_token']
    while token:
        all_samples.append(nusc.get('sample', token))
        token = nusc.get('sample', token)['next']

random.seed(42)
random.shuffle(all_samples)
split_idx = int(len(all_samples) * 0.8)
splits = {'train': all_samples[:split_idx], 'val': all_samples[split_idx:]}
print(f"Total: {len(all_samples)} → train: {split_idx}, val: {len(all_samples) - split_idx}")

def project_box_to_polygon(box, intrinsic, img_w, img_h):
    """
    Project 3D box corners to image plane, return convex hull polygon
    in YOLO normalized format, or None if invalid.
    """
    # Get all 8 corners of the 3D box
    corners_3d = box.corners()  # shape (3, 8)

    # Project to image plane
    corners_2d = view_points(corners_3d, intrinsic, normalize=True)  # (3, 8)

    # Check all corners are in front of camera
    if any(corners_2d[2] < 0):
        return None

    xs = corners_2d[0]
    ys = corners_2d[1]

    # Filter corners that fall within or near the image
    valid = (xs >= -img_w * 0.1) & (xs <= img_w * 1.1) & \
            (ys >= -img_h * 0.1) & (ys <= img_h * 1.1)

    if valid.sum() < MIN_POLYGON_PTS:
        return None

    xs_valid = xs[valid]
    ys_valid = ys[valid]

    # Compute convex hull of valid projected corners
    pts = np.stack([xs_valid, ys_valid], axis=1)

    try:
        if len(pts) >= 3:
            hull = ConvexHull(pts)
            hull_pts = pts[hull.vertices]
        else:
            hull_pts = pts
    except Exception:
        # Degenerate case (collinear points) — use bounding rect instead
        x1, x2 = xs_valid.min(), xs_valid.max()
        y1, y2 = ys_valid.min(), ys_valid.max()
        hull_pts = np.array([[x1,y1],[x2,y1],[x2,y2],[x1,y2]])

    # Clip polygon points to image bounds
    hull_pts[:, 0] = np.clip(hull_pts[:, 0], 0, img_w)
    hull_pts[:, 1] = np.clip(hull_pts[:, 1], 0, img_h)

    # Check polygon area is reasonable
    w_span = hull_pts[:, 0].max() - hull_pts[:, 0].min()
    h_span = hull_pts[:, 1].max() - hull_pts[:, 1].min()
    area_ratio = (w_span * h_span) / (img_w * img_h)

    if area_ratio < MIN_AREA_RATIO or area_ratio > MAX_AREA_RATIO:
        return None

    # Normalize to [0, 1]
    hull_pts[:, 0] /= img_w
    hull_pts[:, 1] /= img_h

    return hull_pts


# Convert all samples
converted = 0
total_polygons = 0
skipped_behind = 0
skipped_class = 0
skipped_area = 0

for split, samples in splits.items():
    for sample in samples:
        cam_token = sample['data']['CAM_FRONT']
        cam_data  = nusc.get('sample_data', cam_token)
        img_path  = os.path.join(NUSCENES_ROOT, cam_data['filename'])
        img_w, img_h = cam_data['width'], cam_data['height']

        # Camera calibration
        cs = nusc.get('calibrated_sensor', cam_data['calibrated_sensor_token'])
        ep = nusc.get('ego_pose', cam_data['ego_pose_token'])
        intrinsic = np.array(cs['camera_intrinsic'])

        boxes = nusc.get_boxes(cam_token)
        labels = []

        for box in boxes:
            # Map to our classes
            cat_simple = next((k for k in CLASS_MAP if k in box.name), None)
            if cat_simple is None:
                skipped_class += 1
                continue

            # Transform: global → ego → camera frame
            box.translate(-np.array(ep['translation']))
            box.rotate(Quaternion(ep['rotation']).inverse)
            box.translate(-np.array(cs['translation']))
            box.rotate(Quaternion(cs['rotation']).inverse)

            # Skip objects behind camera
            if box.center[2] < 0.1:
                skipped_behind += 1
                continue

            # Project to polygon
            polygon = project_box_to_polygon(box, intrinsic, img_w, img_h)
            if polygon is None:
                skipped_area += 1
                continue

            # Format: class x1 y1 x2 y2 x3 y3 ... (YOLO seg format)
            coords = ' '.join([f"{x:.6f} {y:.6f}" for x, y in polygon])
            labels.append(f"{CLASS_MAP[cat_simple]} {coords}")
            total_polygons += 1

        # Symlink image
        img_name = Path(img_path).name
        dst_img = f'{SEG_DIR}/images/{split}/{img_name}'
        if not os.path.exists(dst_img):
            os.symlink(img_path, dst_img)

        # Write label
        with open(f'{SEG_DIR}/labels/{split}/{Path(img_name).stem}.txt', 'w') as f:
            f.write('\n'.join(labels))

        converted += 1

print(f"\n✅ Done: {converted} images, {total_polygons} polygons")
print(f"   Skipped → behind camera: {skipped_behind}, bad area: {skipped_area}, unknown class: {skipped_class}")

# Verify quality
print("\n--- Quality check ---")
for split in ['train', 'val']:
    files  = list(Path(f'{SEG_DIR}/labels/{split}').glob('*.txt'))
    filled = sum(1 for f in files if f.stat().st_size > 0)
    print(f"  {split}: {len(files)} files, {filled} non-empty")

# Show a sample polygon
sample_f = next(f for f in Path(f'{SEG_DIR}/labels/train').glob('*.txt') if f.stat().st_size > 0)
first_line = open(sample_f).readline().strip().split()
cls_id = first_line[0]
n_points = (len(first_line) - 1) // 2
print(f"\nSample: class={cls_id}, polygon has {n_points} points (rectangle=4, convex hull=4–8)")
print(' '.join(first_line[:9]), '...')

In [ ]:
# Create yaml file
seg_yaml = f"""
path: {SEG_DIR}
train: images/train
val: images/val

names:
  0: car
  1: pedestrian
  2: bicycle
  3: motorcycle
  4: bus
  5: truck
  6: traffic_cone
  7: barrier
"""

seg_yaml_path = f'{SEG_DIR}/nuscenes_seg.yaml'
with open(seg_yaml_path, 'w') as f:
    f.write(seg_yaml)
print(f"✅ YAML saved → {seg_yaml_path}")

In [ ]:
# Train YOLO26 Segmentation Model
from ultralytics import YOLO

seg_yaml_path = '/content/drive/MyDrive/yolo_runs/nuscenes_yolo_seg/nuscenes_seg.yaml'

model = YOLO("yolo26n-seg.pt")
model.train(
    data=seg_yaml_path,
    epochs=100,
    imgsz=640,
    batch=16,
    name="yolo26n_seg_nuscenes",
    project=RUNS_DIR,
    device=0,
    workers=2,
    patience=20,
    save=True,
    plots=True,
)

In [ ]:
# Train YOLOv8 Detection Model (baseline)
from ultralytics import YOLO

yaml_path = '/content/drive/MyDrive/yolo_runs/nuscenes_yolo_seg/nuscenes_seg.yaml'
model = YOLO("yolov8n.pt")

model.train(
    data=yaml_path,
    epochs=100,
    imgsz=640,
    batch=16,
    name="yolov8n_nuscenes",
    project=RUNS_DIR,
    device=0,
    workers=2,
    patience=20,
    save=True,
    plots=True,
)

In [ ]:
# Export NuScenes all scenes videos (CAM_FRONT)
import cv2
import os
from nuscenes.nuscenes import NuScenes
from pathlib import Path

NUSCENES_ROOT = '/data/sets/nuscenes'
VIDEO_DIR = '/content/videos'
Path(VIDEO_DIR).mkdir(exist_ok=True)

nusc = NuScenes(version='v1.0-mini', dataroot=NUSCENES_ROOT, verbose=False)

for i, scene in enumerate(nusc.scene):
    out_path = f'{VIDEO_DIR}/scene_{i:02d}_{scene["name"]}.mp4'

    # Start from first SAMPLE (keyframe)
    sample_token = scene['first_sample_token']
    sample = nusc.get('sample', sample_token)

    # Get first cam_front sample_data token
    cam_sd_token = sample['data']['CAM_FRONT']

    # Collect ALL sample_data tokens (keyframes + sweeps) in order
    all_frames = []
    current_token = cam_sd_token
    while current_token:
        sd = nusc.get('sample_data', current_token)
        all_frames.append(sd)
        current_token = sd['next']

    # Write video
    writer = None
    for sd in all_frames:
        img_path = os.path.join(NUSCENES_ROOT, sd['filename'])
        if not os.path.exists(img_path):
            continue
        frame = cv2.imread(img_path)
        if frame is None:
            continue
        if writer is None:
            h, w = frame.shape[:2]
            writer = cv2.VideoWriter(
                out_path,
                cv2.VideoWriter_fourcc(*'mp4v'),
                12, (w, h)
            )
        writer.write(frame)

    if writer:
        writer.release()
    print(f"✅ Scene {i:02d}: {len(all_frames)} frames → {Path(out_path).name}")

print(f"\nAll videos saved to {VIDEO_DIR}")

In [ ]:
# YOLO ultralytics solutions demo test on CAM_FRONT scenes videos[2]
import cv2
import os
from ultralytics import solutions
from pthlib import Path

MODEL = '/content/drive/MyDrive/yolo_runs/yolo26n_nuscenes/weights/best.pt'
VIDEO_DIR = '/content/videos'
videos = sorted(os.listdir(VIDEO_DIR))
VIDEO = f'{VIDEO_DIR}/{videos[2]}'
OUTPUT_DIR = '/content/solutions_output'
Path(OUTPUT_DIR).mkdir(exist_ok=True)

def get_video_props(path):
    cap = cv2.VideoCapture(path)
    w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 12
    cap.release()
    return w, h, fps

W, H, FPS = get_video_props(VIDEO)
print(f"Video: {VIDEO}")
print(f"Size: {W}x{H} @ {FPS:.1f} FPS")

# ── 1. Speed Estimation ──────────────────────────────────────────
print("\nRunning Speed Estimator...")
cap = cv2.VideoCapture(VIDEO)
speed_obj = solutions.SpeedEstimator(model=MODEL, show=False)
out = cv2.VideoWriter(f'{OUTPUT_DIR}/speed.mp4',
                      cv2.VideoWriter_fourcc(*'mp4v'), FPS, (W, H))
while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    out.write(speed_obj.process(frame).plot_im)
cap.release(); out.release()
print("✅ Speed estimation done")

# ── 2. Heatmap ───────────────────────────────────────────────────
print("Running Heatmap...")
cap = cv2.VideoCapture(VIDEO)
heat_obj = solutions.Heatmap(
    model=MODEL,
    colormap=cv2.COLORMAP_PARULA,
    show=False,
)
out = cv2.VideoWriter(f'{OUTPUT_DIR}/heatmap.mp4',
                      cv2.VideoWriter_fourcc(*'mp4v'), FPS, (W, H))
while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    out.write(heat_obj.process(frame).plot_im)
cap.release(); out.release()
print("✅ Heatmap done")

# ── 3. Object Counter ────────────────────────────────────────────
print("Running Object Counter...")
cap = cv2.VideoCapture(VIDEO)
count_obj = solutions.ObjectCounter(
    model=MODEL,
    region=[(0, int(H*0.65)), (W, int(H*0.65))],
    show=False,
)
out = cv2.VideoWriter(f'{OUTPUT_DIR}/counting.mp4',
                      cv2.VideoWriter_fourcc(*'mp4v'), FPS, (W, H))
while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    out.write(count_obj.process(frame).plot_im)
cap.release(); out.release()
print("✅ Object counting done")

# ── 4. Analytics ─────────────────────────────────────────────────
print("Running Analytics...")
cap = cv2.VideoCapture(VIDEO)
analytics_obj = solutions.Analytics(
    model=MODEL,
    analytics_type="line",
    show=False,
)
out = cv2.VideoWriter(f'{OUTPUT_DIR}/analytics.mp4',
                      cv2.VideoWriter_fourcc(*'mp4v'), FPS, (W, H))
frame_idx = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    frame_idx += 1
    out.write(analytics_obj.process(frame, frame_idx).plot_im)
cap.release(); out.release()
print("✅ Analytics done")

# ── 5. Segmentation masks ────────────────────────────────────────
print("Running Segmentation masks...")
from ultralytics import YOLO
MODEL_SEG = '/content/drive/MyDrive/yolo_runs/yolo26n_seg_nuscenes/weights/best.pt'
seg_model = YOLO(MODEL_SEG)
cap = cv2.VideoCapture(VIDEO)
out = cv2.VideoWriter(f'{OUTPUT_DIR}/segmentation_masks.mp4',
                      cv2.VideoWriter_fourcc(*'mp4v'), FPS, (W, H))
while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    out.write(seg_model(frame, verbose=False)[0].plot())
cap.release(); out.release()
print("✅ Segmentation masks done")

print(f"\nAll outputs saved to {OUTPUT_DIR}")

In [ ]:
import cv2
import os
from ultralytics import solutions
from pathlib import Path

MODEL = '/content/drive/MyDrive/yolo_runs/yolo26n_nuscenes/weights/best.pt'
VIDEO_DIR = '/content/videos'
videos = sorted(os.listdir(VIDEO_DIR))
VIDEO = f'{VIDEO_DIR}/{videos[2]}'  # scene 2, stopped at traffic light
OUTPUT_DIR = '/content/solutions_output'
Path(OUTPUT_DIR).mkdir(exist_ok=True)

W = int(cv2.VideoCapture(VIDEO).get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cv2.VideoCapture(VIDEO).get(cv2.CAP_PROP_FRAME_HEIGHT))
FPS = cv2.VideoCapture(VIDEO).get(cv2.CAP_PROP_FPS) or 12

print(f"Using: {videos[2]}")
print(f"meter_per_pixel = 0.0040 (calibrated from nuScenes intrinsics at ~10m depth)")

print("\nRunning Speed Estimator (calibrated)...")
cap = cv2.VideoCapture(VIDEO)
speed_obj = solutions.SpeedEstimator(
    model=MODEL,
    show=False,
    meter_per_pixel=0.0040,
    classes=[0, 1, 2, 3, 4, 5],  # car, pedestrian, bicycle, motorcycle, bus, truck
                                   # excludes: 6=traffic_cone, 7=barrier
)
out = cv2.VideoWriter(f'{OUTPUT_DIR}/speed_calibrated.mp4',
                      cv2.VideoWriter_fourcc(*'mp4v'), FPS, (W, H))
while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    out.write(speed_obj.process(frame).plot_im)
cap.release(); out.release()

# Preview middle frame
cap = cv2.VideoCapture(f'{OUTPUT_DIR}/speed_calibrated.mp4')
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.set(cv2.CAP_PROP_POS_FRAMES, total // 2)
ret, frame = cap.read()
cap.release()
cv2.imwrite('/content/speed_preview.jpg', frame)

from IPython.display import Image, display
display(Image('/content/speed_preview.jpg'))
print(f"✅ Done — {total} frames")

In [ ]:
# Segmentation Mask using COCO weights test
from ultralytics import YOLO
import cv2
from pathlib import Path
import os

# Use base COCO weights, not fine-tuned
model = YOLO("yolo26n-seg.pt")

VIDEO_DIR = '/content/videos'
VIDEO = f'{VIDEO_DIR}/{sorted(os.listdir(VIDEO_DIR))[2]}'
OUTPUT_DIR = '/content/solutions_output'

cap = cv2.VideoCapture(VIDEO)
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
FPS = cap.get(cv2.CAP_PROP_FPS) or 12

out = cv2.VideoWriter(f'{OUTPUT_DIR}/segmentation_coco_cone.mp4',
                      cv2.VideoWriter_fourcc(*'mp4v'), FPS, (W, H))

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    out.write(model(frame, verbose=False)[0].plot())

cap.release(); out.release()

# Preview
cap = cv2.VideoCapture(f'{OUTPUT_DIR}/segmentation_coco_cone.mp4')
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.set(cv2.CAP_PROP_POS_FRAMES, total // 2)
ret, frame = cap.read()
cap.release()
cv2.imwrite('/content/seg_coco_preview_cone.jpg', frame)

from IPython.display import Image, display
display(Image('/content/seg_coco_preview_cone.jpg'))
print("✅ Done")